# 07 — Recipes, training knobs, and honest evaluation

Everything OrientedDet trains is described by a JSON **recipe**. This lesson reads one, resolves its inheritance, patches and saves a copy ready for `odet train`, then turns to the other half of an experiment: how detection is scored, how to compute mAP yourself, and why the number you quote must come from the official test server.

| | |
|---|---|
| Needs | Internet (Hub checkpoint); sections 5–6 need a local checkout with `demo/dota_samples/` |
| Kaggle | Sections 1–4 and 7. The DOTA tiles are academic-use only. |
| Posts | [Lessons learned on DOTA and MMRotate parity](https://deeplearning.earth/posts/2026-09-17_lessons_learned_on_dota_oriented_det_and_mmrotate_parity/) · [v0.3.0: four datasets and ONNX](https://deeplearning.earth/posts/2026-09-21_oriented-det_v0_3_0_four_datasets_and_onnx/) |
| Docs | [Configuration guide](https://github.com/DL4EO/oriented-det/blob/main/docs/user-guide/configuration.md) · [Training guide](https://github.com/DL4EO/oriented-det/blob/main/docs/user-guide/training.md) |

## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det>=0.3.1"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}"

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd), flush=True)
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input/<dataset> whose name contains one of `needles` (None off Kaggle)."""
    root = Path("/kaggle/input")
    dirs = sorted(p for p in root.iterdir() if p.is_dir()) if root.is_dir() else []
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 01 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")

## 1. Anatomy of a recipe

A recipe lists `_base_` fragments (dataset, model, schedule, preprocessing), then overrides whatever it needs. The sections you will touch most:

| Section | Controls |
|---|---|
| `dataset` | format (`dota`, `hrsc2016`, `ssdd`, ...), paths, sample caps, difficult-object handling |
| `model` | architecture, losses (lesson 03), RPN/RoI sampling, NMS |
| `training` | epochs, learning rate and schedule, AMP, gradient clipping |
| `preprocessing` | canvas size, resize mode, normalisation, flips, rotation, mosaic |
| `checkpoint` | where to initialise from (`hf://...` for fine-tuning) and what to resume |
| `evaluation` | IoU for mAP, how often to compute it during training |
| `production` | deploy defaults baked into the checkpoint (lesson 01) |

In [ ]:
import json

RECIPE = CONFIGS / "rotated_faster_rcnn/dota_le90_1x.json"
raw = json.loads(RECIPE.read_text())
print(json.dumps({k: raw[k] for k in ("_base_", "training", "dataset")}, indent=2))

## 2. Resolve the inheritance

`TrainingExperimentConfig.load` merges the bases in order, then the recipe on top (later wins, dictionaries merge key by key). The result is a typed object with defaults for everything left unset.

In [ ]:
from oriented_det.train.config import TrainingExperimentConfig

for rel in raw["_base_"]:
    base = json.loads((RECIPE.parent / rel).read_text())
    print(f"{rel:48s} provides {', '.join(k for k in base if k != '_base_')}")

cfg = TrainingExperimentConfig.load(RECIPE)
print("\nresolved:")
print("  model_type     ", cfg.model_type, "| backbone frozen stages", cfg.model.frozen_stages)
print("  schedule       ", cfg.training.num_epochs, "epochs, lr", cfg.training.learning_rate,
      "decayed at", cfg.training.lr_scheduler_milestones)
print("  canvas         ", cfg.preprocessing.resize_mode, cfg.preprocessing.target_size,
      "| flips h/v/diag", cfg.preprocessing.enable_flip_horizontal, cfg.preprocessing.enable_flip_vertical,
      cfg.preprocessing.enable_flip_diagonal)
print("  train tiles    ", cfg.dataset.train_tiles_dirs)
print("  val tiles      ", cfg.dataset.val_tiles_dirs)

Note the last two lines: the public DOTA recipes train on **train + val** and validate on **val**. That is deliberate (more data for the checkpoint you submit to the test server) and it makes the in-training validation mAP optimistic. Keep that in mind for section 5.

## 3. Patch, save, train

The usual workflow: load a published recipe, change a few fields, save it next to your experiment, and point `odet train` at it. Here: a smoke run on 64 tiles for one epoch.

In [ ]:
from pathlib import Path

DOTA_TILED = Path(os.environ.get("DOTA_TILED_ROOT", "/path/to/data/DOTA-v1.0-tiled"))

cfg = TrainingExperimentConfig.load(RECIPE)
cfg.dataset.data_root = DOTA_TILED
cfg.dataset.train_tiles_dirs = [str(DOTA_TILED / "train")]
cfg.dataset.val_tiles_dirs = [str(DOTA_TILED / "val")]
cfg.dataset.max_train_samples = 64
cfg.dataset.max_val_samples = 16
cfg.training.num_epochs = 1
my_recipe = WORK / "07_my_recipe.json"
cfg.save(my_recipe)

reloaded = TrainingExperimentConfig.load(my_recipe)
print("saved", my_recipe)
print("round trip:", reloaded.dataset.max_train_samples, "tiles,", reloaded.training.num_epochs, "epoch")
print("train with:  odet train --config", my_recipe)

`odet train` writes everything under `runs/<model_type>/<timestamp>/`: checkpoints (`best_mAP_*.pth`, `checkpoint_epoch_NNNN.pth`), a frozen copy of the resolved config, TensorBoard logs, and periodic mAP. The setup cell points `ORIENTED_DET_PROJECT_ROOT` at the work directory, so notebook runs land in `work_dirs/runs/`. Lessons 08–10 run real fine-tunes; the switches you will see there are `checkpoint.load_from_checkpoint = "hf://..."` to start from a Hub model and `dataset.max_*_samples` to keep smoke runs short.

## 4. How detection is scored

1. Keep **all** detections down to a low score (0.05). mAP is computed over the full ranking, so a high threshold throws information away.
2. Per class, sort detections by score. Walk down the list: a detection is a **true positive** if it overlaps an unmatched label with rotated IoU ≥ 0.5, otherwise a false positive. Labels marked *difficult* neither count nor penalise.
3. That walk traces a precision–recall curve; **AP** is the area under it (VOC style). **mAP50** is the mean AP over classes.

Two DOTA numbers get confused all the time:

- **Official Task 1** — predictions on the hidden test set, scored by the DOTA server. This is the number to quote (76.73% for `oriented_rcnn_dota_le90_1x`).
- **Eval-val** — the same metric on the val tiles. Because the recipes train on val (section 2), it is **leaky** and always higher. It is useful for comparing runs that share the leak, never as the model's score.

## 5. Compute mAP yourself

The 36 sample tiles in `demo/dota_samples/` come from DOTA val. Compute mAP on them twice: once with every detection down to 0.05, once only above the deploy floor.

In [ ]:
from PIL import Image
from oriented_det.data import Detection, GroundTruth, compute_oriented_map
from oriented_det.data.dota import DOTAAnnotation

SAMPLES = REPO / "demo" / "dota_samples" if HAS_REPO else None
if SAMPLES is None or not SAMPLES.is_dir():
    SAMPLES = None
    print("demo/dota_samples/ needs a local checkout (DOTA academic licence) - skipping sections 5-6.")

model, config, class_names = hub_model("oriented_rcnn_dota_le90_1x")
dets, gts = {}, {}
for img in sorted(SAMPLES.glob("*.jpg")) if SAMPLES else []:
    lines = [l for l in img.with_suffix(".txt").read_text().splitlines() if l.strip()]
    gts[img.stem] = [GroundTruth(a.rbox, class_names.index(a.class_name), a.class_name, a.difficult, img.stem)
                     for a in map(DOTAAnnotation.from_line, lines)]
    dets[img.stem] = [Detection(d["rbox"], float(d["score"]), d["label"] - 1, class_names[d["label"] - 1], img.stem)
                      for d in detect(model, config, class_names, Image.open(img).convert("RGB"), score_thr=0.05)]

if gts:
    floor = config.production.score_threshold
    full, per_class, _ = compute_oriented_map(dets, gts, class_names=class_names, show_progress=False)
    cut, _, _ = compute_oriented_map({k: [d for d in v if d.score >= floor] for k, v in dets.items()}, gts,
                                     class_names=class_names, show_progress=False)
    print(f"{len(gts)} tiles, {sum(map(len, gts.values()))} labels")
    print(f"mAP50 with detections >= 0.05:      {100 * full:.2f}%")
    print(f"mAP50 with detections >= {floor} only: {100 * cut:.2f}%")
    print("official Task 1 for this checkpoint: 76.73%")
    order = sorted(per_class, key=per_class.get)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.barh(order, [100 * per_class[c] for c in order]); ax.set_xlabel("AP50 on the 36 val tiles (%)")
    plt.tight_layout(); plt.show()

Two lessons in two lines. Cutting at the deploy floor lowers mAP because the tail of the precision–recall curve disappears: evaluate low, deploy high. And the 36-tile number is far above the official 76.73% because the model trained on these very tiles; add 36 tiles' worth of sampling noise and it says almost nothing about test performance.

## 6. The same with `odet preds`

For real validation sets, `odet preds` runs inference over a folder (sliding windows included), writes `predictions.json`, and computes mAP, precision–recall sweeps, per-class best thresholds, and a markdown report. `--metrics-from-json` recomputes the metrics later without touching the GPU. It expects `images/` and `labels/` sub-folders, so link the samples into that layout first.

In [ ]:
if SAMPLES:
    mini_val = WORK / "07_mini_val"
    for sub, pattern in (("images", "*.jpg"), ("labels", "*.txt")):
        (mini_val / sub).mkdir(parents=True, exist_ok=True)
        for f in SAMPLES.glob(pattern):
            link = mini_val / sub / f.name
            if not link.exists():
                link.symlink_to(f.resolve())
    ckpt, sidecar = hub_files("oriented_rcnn_dota_le90_1x")
    preds_dir = WORK / "07_preds"
    run([*ODET, "preds", "--checkpoint", ckpt, "--config", sidecar,
         "--val-dir", mini_val, "--data-split", "val", "--score-threshold", 0.05,
         "--no-diagnostics", "--output-dir", preds_dir])
    run([*ODET, "preds", "--metrics-from-json", preds_dir])
    print(sorted(p.name for p in preds_dir.iterdir()))

`odet preds` reports 91.10% where section 5 found 91.62%, on the same detections. The recipe's `dataset.difficult_strategy` is `"keep"`, which counts the 36 *difficult* labels like any other, while section 5 passed the flag through and the evaluator ignored them, as the DOTA devkit does. Small evaluation settings move mAP; always compare numbers produced the same way. (The metrics step takes a few minutes: exact polygon IoU on the CPU for ~3,000 boxes.)

On a full training run the same two commands take an experiment directory instead: `odet preds --experiment-dir runs/<model>/<timestamp> --data-split val`, or simply `make eval-val EXPERIMENT=runs/<model>/<timestamp>` in a checkout. For the official score, `odet dota-submit` writes the Task 1 submission zip for the DOTA server.

## 7. Read the published reports

Every Hub checkpoint has a committed `docs/eval-reports/<slug>/model_analysis.md` with both numbers. Put them side by side:

In [ ]:
import re


def eval_report(slug):
    rel = f"docs/eval-reports/{slug}/model_analysis.md"
    path = REPO / rel if HAS_REPO else WORK / rel
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/{rel}", path)
    return path.read_text(encoding="utf-8")


print(f"{'checkpoint':34s} {'Task 1 (quote this)':>20s} {'eval-val (leaky)':>17s}")
for slug in ["oriented_rcnn_dota_le90_1x", "rotated_faster_rcnn_dota_le90_1x", "rotated_fcos_dota_le90_1x",
             "rotated_retinanet_dota_le90_1x", "oriented_rcnn_dota_le90_3x", "rotated_faster_rcnn_dota_le90_3x"]:
    text = eval_report(slug)
    task1 = re.search(r"Task 1 \*\*([0-9.]+)%\*\*", text)[1]
    leaky = 100 * float(re.search(r"mAP50: `([0-9.]+)`", text)[1])
    print(f"{slug:34s} {task1 + '%':>20s} {leaky:16.2f}%")

The gap grows with training length. The 3× schedules see the val tiles three times as often, so their eval-val climbs while Task 1 stays flat. The DOTA lessons post walks through how that fooled early comparisons.

## What to try next

- Change `evaluation.iou_threshold` to 0.75 in a copy of the recipe: "AP75" rewards precise boxes and separates the detectors much more than AP50.
- Rerun section 5 with `rotated_fcos_dota_le90_1x`. Does evaluating at FCOS's 0.20 floor cost more or less mAP than for Oriented R-CNN?
- Open `work_dirs/07_preds/` and read the generated report and PR curves.
- **Next:** [08 — HRSC2016 ship finetune](08_hrsc2016_finetune.ipynb): the first real training run.